In [1]:
import pandas as pd
import numpy as np
import missingno as msno

In [2]:
cities = [
    'Anaheim', 'Bakersfield-California', 'Calexico-Ethel Street',
    'Chico-East Avenue', 'Fresno - Garland', 'Keeler',
    'Los Angeles-North Main Street', 'Mira Loma (Van Buren)',
    'Oakland', 'Rubidoux', 'Salinas 3', 'Simi Valley-Cochran Street',
    'Temecula', 'Thousand Oaks', 'San_Jose_-_Jackson'
]

city = 'San Jose - Jackson'

### Pollution Data


In [3]:
df1 = pd.read_csv('Data/top_40_cities_data.csv')
df1['Date'] = pd.to_datetime(df1['Date'])
df1 = df1.sort_values(['Date'])
df1 = df1[df1['Name'] == city]
df1 = df1.reset_index(drop=True)


df1 = df1.rename(columns={
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'PM2.5': 'pm2.5',
    'Ozone': 'Ozone',
    'AQI': 'aqi',
    'Name': 'city',
    'Date': 'date'
})


df1.set_index(['date'], inplace=True)
df1.head()

,latitude,longitude,pm2.5,Ozone,aqi,city
date,,,,,,
2014-01-01,37.348497,-121.894898,60.4,0.015,154,San Jose - Jackson
2014-01-02,37.348497,-121.894898,33.0,0.010,96,San Jose - Jackson
2014-01-03,37.348497,-121.894898,32.6,0.010,95,San Jose - Jackson
2014-01-04,37.348497,-121.894898,26.4,0.015,83,San Jose - Jackson
2014-01-05,37.348497,-121.894898,19.5,0.018,70,San Jose - Jackson


### Weather Data

In [4]:
# df2 = pd.read_csv('Data/Weather/historical_weather_data_Anaheim.csv')
df2 = pd.read_csv('Data/Weather/historical_weather_data_San_Jose_-_Jackson.csv')

df2['date'] = pd.to_datetime(df2['date'])
df2.set_index('date', inplace=True)

df2 = (
    df2.groupby('City')
    .resample('D')
    .mean()       
    .reset_index()
)
df2.set_index(['date'], inplace=True)
df2.drop(['City'], inplace=True,axis=1)


df2.head()

,temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,precipitation,cloud_cover
date,,,,,,
2014-01-01,10.993118,42.393568,8.427079,1021.341176,0.0,31.823529
2014-01-02,12.511500,41.723330,8.878908,1019.933333,0.0,44.041667
2014-01-03,14.938583,37.342059,7.806972,1017.075000,0.0,92.541667
2014-01-04,14.390667,40.825293,6.490594,1015.345833,0.0,55.375000
2014-01-05,11.257333,47.339449,7.805876,1018.612500,0.0,0.000000


### Merge both Weather and Pollution Data (Inner Join)

In [5]:
df = pd.merge(df1, df2, left_index=True, right_index=True, how='inner')
df.reset_index(inplace=True)
df.head()

,date,latitude,longitude,pm2.5,Ozone,aqi,city,temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,precipitation,cloud_cover
0,2014-01-01,37.348497,-121.894898,60.4,0.015,154,San Jose - Jackson,10.993118,42.393568,8.427079,1021.341176,0.0,31.823529
1,2014-01-02,37.348497,-121.894898,33.0,0.010,96,San Jose - Jackson,12.511500,41.723330,8.878908,1019.933333,0.0,44.041667
2,2014-01-03,37.348497,-121.894898,32.6,0.010,95,San Jose - Jackson,14.938583,37.342059,7.806972,1017.075000,0.0,92.541667
3,2014-01-04,37.348497,-121.894898,26.4,0.015,83,San Jose - Jackson,14.390667,40.825293,6.490594,1015.345833,0.0,55.375000
4,2014-01-05,37.348497,-121.894898,19.5,0.018,70,San Jose - Jackson,11.257333,47.339449,7.805876,1018.612500,0.0,0.000000


### Getting the full dates range and merging 

In [6]:
date_range = pd.date_range(start='2014-01-01', end='2025-04-04', freq='D')
full_dates_df = pd.DataFrame(date_range, columns=['date'])
df = pd.merge(full_dates_df, df, on='date', how='left')
df.head()
full_dates_df["date"] = full_dates_df["date"].astype("datetime64[ns]")  
df["date"] = df["date"].astype("datetime64[ns]")  

df = pd.merge(full_dates_df, df, on="date", how="left")  
df.head()

,date,latitude,longitude,pm2.5,Ozone,aqi,city,temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,precipitation,cloud_cover
0,2014-01-01,37.348497,-121.894898,60.4,0.015,154.0,San Jose - Jackson,10.993118,42.393568,8.427079,1021.341176,0.0,31.823529
1,2014-01-02,37.348497,-121.894898,33.0,0.010,96.0,San Jose - Jackson,12.511500,41.723330,8.878908,1019.933333,0.0,44.041667
2,2014-01-03,37.348497,-121.894898,32.6,0.010,95.0,San Jose - Jackson,14.938583,37.342059,7.806972,1017.075000,0.0,92.541667
3,2014-01-04,37.348497,-121.894898,26.4,0.015,83.0,San Jose - Jackson,14.390667,40.825293,6.490594,1015.345833,0.0,55.375000
4,2014-01-05,37.348497,-121.894898,19.5,0.018,70.0,San Jose - Jackson,11.257333,47.339449,7.805876,1018.612500,0.0,0.000000


In [7]:

df.isna().sum()
df_clean = df.dropna()
df_clean = df.fillna(method='ffill')  
df.head()

/var/folders/yb/3k_z9pjn63g2ms9dcj0h0bch0000gn/T/ipykernel_29164/460745309.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean = df.fillna(method='ffill')


,date,latitude,longitude,pm2.5,Ozone,aqi,city,temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,precipitation,cloud_cover
0,2014-01-01,37.348497,-121.894898,60.4,0.015,154.0,San Jose - Jackson,10.993118,42.393568,8.427079,1021.341176,0.0,31.823529
1,2014-01-02,37.348497,-121.894898,33.0,0.010,96.0,San Jose - Jackson,12.511500,41.723330,8.878908,1019.933333,0.0,44.041667
2,2014-01-03,37.348497,-121.894898,32.6,0.010,95.0,San Jose - Jackson,14.938583,37.342059,7.806972,1017.075000,0.0,92.541667
3,2014-01-04,37.348497,-121.894898,26.4,0.015,83.0,San Jose - Jackson,14.390667,40.825293,6.490594,1015.345833,0.0,55.375000
4,2014-01-05,37.348497,-121.894898,19.5,0.018,70.0,San Jose - Jackson,11.257333,47.339449,7.805876,1018.612500,0.0,0.000000


In [8]:
#msno.matrix(df)

### Fill missing values

In [9]:
df = df.ffill().bfill()
df.head()

,date,latitude,longitude,pm2.5,Ozone,aqi,city,temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,precipitation,cloud_cover
0,2014-01-01,37.348497,-121.894898,60.4,0.015,154.0,San Jose - Jackson,10.993118,42.393568,8.427079,1021.341176,0.0,31.823529
1,2014-01-02,37.348497,-121.894898,33.0,0.010,96.0,San Jose - Jackson,12.511500,41.723330,8.878908,1019.933333,0.0,44.041667
2,2014-01-03,37.348497,-121.894898,32.6,0.010,95.0,San Jose - Jackson,14.938583,37.342059,7.806972,1017.075000,0.0,92.541667
3,2014-01-04,37.348497,-121.894898,26.4,0.015,83.0,San Jose - Jackson,14.390667,40.825293,6.490594,1015.345833,0.0,55.375000
4,2014-01-05,37.348497,-121.894898,19.5,0.018,70.0,San Jose - Jackson,11.257333,47.339449,7.805876,1018.612500,0.0,0.000000
